# 03 — Worst-case hypercube (analytical + ngspice)

Endpoint **hypercube** on the envelope sources $V_{\mathrm{ADJ}}$, $I_{\mathrm{ADJ}}$,
$R_1$, $R_2$ ($2^4 = 16$ corners), plus RSS of one-at-a-time half-spans.

This notebook is the regulator analog of the comparator $V_L$/$V_H$ WC chapter:

| Picture | Comparator | This regulator |
|---------|------------|----------------|
| Range bars | $V_L$, $V_H$, $V_{\mathrm{HYS}}$ min…max | $V_{\mathrm{OUT}}$, headroom, $P_{\mathrm{diss}}$, efficiency |
| Green points | one $(V_L,V_H)$ per hypercube corner | one $({\mathrm{headroom}}, V_{\mathrm{OUT}})$ per corner |
| Envelope box | axis-aligned min/max in the $V_L$–$V_H$ plane | same, in headroom–$V_{\mathrm{OUT}}$ |
| RSS hatch | $\sqrt{\sum \delta_i^2}$ inside the box | same formula on $V_{\mathrm{OUT}}$ |

The hypercube is **pessimistic** (every part at a worst endpoint together).
RSS is the usual independent-error bound.

$$
\mathrm{RSS}(m)=\sqrt{\sum_i \delta_i(m)^2},\qquad
\delta_i(m)=\tfrac12\bigl|m(x_i^{\max})-m(x_i^{\min})\bigr|
$$

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


## Walk the 16-corner envelope

Signs: $R_1$ lo/hi × $R_2$ lo/hi × $V_{\mathrm{ADJ}}$ lo/hi × $I_{\mathrm{ADJ}}$ lo/hi.
$V_{\mathrm{OUT}}$ rises with $V_{\mathrm{ADJ}}$, $I_{\mathrm{ADJ}}$, $R_1$ and falls with $R_2$.

In [2]:
from p5v_design.worst_case import (
    analytical_worst_case, oat_contributions, rss_bounds, rss_bounds_many,
    envelope_worst_case, spice_wc_corners, corners_frame,
)
from p5v_design.simulate import find_ngspice

wc = analytical_worst_case(p, over_temp=False)
wc_ot = analytical_worst_case(p, over_temp=True)
rss = rss_bounds_many(p, over_temp=False)
oat = oat_contributions(p, metrics=("vout", "headroom", "p_diss", "efficiency"))
cdf = corners_frame(wc)
print("16 corners + nominal rows:", len(wc.rows))
display(pd.DataFrame([wc.nominal, wc.minimum, wc.maximum], index=["nom", "min", "max"]).T)
display(cdf.sort_values("vout")[["corner", "r1", "r2", "vadj", "iadj", "vout", "headroom", "p_diss"]].head(8))
display(cdf.sort_values("vout")[["corner", "vout", "headroom"]].tail(8))
print("RSS Vout", rss["vout"])

16 corners + nominal rows: 17


,nom,min,max
vout,5.000000e+00,4.925000,5.075000
gain,1.000000e+00,1.000000,1.000000
headroom,6.700000e+00,6.625000,6.775000
vdo,3.000000e-01,0.300000,0.300000
ignd,1.800000e-03,0.001800,0.001800
iin,5.180000e-02,0.051800,0.051800
p_diss,3.716000e-01,0.367850,0.375350
efficiency,4.021879e-01,0.396155,0.408221
r1,0.000000e+00,NaN,NaN
r2,1.000000e+12,NaN,NaN


,corner,r1,r2,vadj,iadj,vout,headroom,p_diss
0,R1lo_R2lo_Valo_Ialo,0.0,9.900000e+11,4.925,0.0,4.925,6.775,0.37535
1,R1lo_R2lo_Valo_Iahi,0.0,9.900000e+11,4.925,0.0,4.925,6.775,0.37535
4,R1lo_R2hi_Valo_Ialo,0.0,1.010000e+12,4.925,0.0,4.925,6.775,0.37535
5,R1lo_R2hi_Valo_Iahi,0.0,1.010000e+12,4.925,0.0,4.925,6.775,0.37535
8,R1hi_R2lo_Valo_Ialo,0.0,9.900000e+11,4.925,0.0,4.925,6.775,0.37535
9,R1hi_R2lo_Valo_Iahi,0.0,9.900000e+11,4.925,0.0,4.925,6.775,0.37535
12,R1hi_R2hi_Valo_Ialo,0.0,1.010000e+12,4.925,0.0,4.925,6.775,0.37535
13,R1hi_R2hi_Valo_Iahi,0.0,1.010000e+12,4.925,0.0,4.925,6.775,0.37535


,corner,vout,headroom
2,R1lo_R2lo_Vahi_Ialo,5.075,6.625
3,R1lo_R2lo_Vahi_Iahi,5.075,6.625
6,R1lo_R2hi_Vahi_Ialo,5.075,6.625
7,R1lo_R2hi_Vahi_Iahi,5.075,6.625
10,R1hi_R2lo_Vahi_Ialo,5.075,6.625
11,R1hi_R2lo_Vahi_Iahi,5.075,6.625
14,R1hi_R2hi_Vahi_Ialo,5.075,6.625
15,R1hi_R2hi_Vahi_Iahi,5.075,6.625


RSS Vout {'nominal': 5.0, 'delta_rss': 0.07500000000000018, 'low': 4.925, 'high': 5.075, 'half_spans': {'Vadj': 0.07500000000000018, 'Iadj': 0.0, 'R1': 0.0, 'R2': 0.0}}


## Range bars — analytical WC (comparator threshold-band chart)

Each bar is **min…max** of the hypercube. Black ticks are the endpoints;
the black circle is nominal. Purple dotted line is the 5 V goal.
Edit `ann_bar` offsets if a label collides.

In [3]:
ann_bar = [
    dict(text="5 V goal", xy=(5.0, 0.0), offset=(8, 18), arrow=True, color="#9467bd"),
]
pwc.plot_range_bars(
    {"analytical": pwc.band_from_wc(wc, "vout")},
    metric="vout", target=5.0, rss=(rss["vout"]["low"], rss["vout"]["high"]),
    annotations=ann_bar, path=FIG/"03_range_vout.png",
    title=r"Analytical WC range  $V_{\mathrm{OUT}}$",
)
plt.show()
pwc.plot_metric_range_grid(
    {
        "vout": {"analytical": pwc.band_from_wc(wc, "vout")},
        "headroom": {"analytical": pwc.band_from_wc(wc, "headroom")},
        "p_diss": {"analytical": pwc.band_from_wc(wc, "p_diss")},
        "efficiency": {"analytical": pwc.band_from_wc(wc, "efficiency")},
    },
    rss_by_metric={k: (rss[k]["low"], rss[k]["high"]) for k in rss},
    target_by_metric={"vout": 5.0},
    path=FIG/"03_range_grid.png",
    title="Analytical WC range bars (four metrics)",
)
plt.show()
pwc.plot_whiskers_engines(
    {"analytical": pwc.band_from_wc(wc, "vout")},
    metric="vout", target=5.0, path=FIG/"03_whiskers_vout.png",
)
plt.show()

cell:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## WC fill vs RSS hatch vs 5 V (1-D band)

Same idea as the comparator transfer plot: **WC** is the wide light band,
**RSS** is the hatched inner band, nominal is the solid line, goal is dotted.

In [4]:
ann_band = [
    dict(text="WC envelope", xy=(wc.minimum["vout"], 0), offset=(10, 16), arrow=True, color="#d62728"),
    dict(text="RSS", xy=(rss["vout"]["low"], 0), offset=(-20, 16), arrow=True, color="#2ca02c"),
]
pwc.plot_wc_rss_band(
    nom=wc.nominal["vout"], wc=(wc.minimum["vout"], wc.maximum["vout"]),
    rss=(rss["vout"]["low"], rss["vout"]["high"]), target=5.0,
    annotations=ann_band, path=FIG/"03_wc_rss_band.png",
)
plt.show()

cell:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Hypercube scatter — each green point is one corner

Comparator notebook 01 plots $(V_L, V_H)$ for every corner. Here the natural
plane is **headroom vs $V_{\mathrm{OUT}}$**: both move when the divider or
$V_{\mathrm{ADJ}}$ hits an endpoint. The dashed box is the axis-aligned envelope.
The diamond is the 5 V goal (at nominal headroom).

In [5]:
ann_sc = [
    dict(text="nominal", xy=(wc.nominal["headroom"], wc.nominal["vout"]), offset=(12, -16), arrow=True),
    dict(text="5 V goal", xy=(wc.nominal["headroom"], 5.0), offset=(14, 10), arrow=True, color="#9467bd"),
]
pwc.plot_hypercube_scatter(
    cdf, x="headroom", y="vout",
    nominal=(wc.nominal["headroom"], wc.nominal["vout"]),
    target_y=5.0, annotations=ann_sc, path=FIG/"03_hypercube_scatter.png",
    title=r"Hypercube corners  (headroom, $V_{\mathrm{OUT}}$)",
)
plt.show()
pwc.plot_hypercube_scatter(
    cdf, x="gain", y="vout",
    nominal=(wc.nominal["gain"], wc.nominal["vout"]),
    target_y=5.0, path=FIG/"03_hypercube_gain.png",
    title=r"Hypercube  $G=1+R_1/R_2$ vs $V_{\mathrm{OUT}}$",
)
plt.show()
pwc.plot_hypercube_params(cdf, path=FIG/"03_hypercube_params.png")
plt.show()

cell:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Stem plot of all 16 corners

Every corner on one axis, sorted in the hypercube walk order. The 5 V line
shows the page-3 output-voltage box with SENSE tied to OUT.

In [6]:
ann_st = [dict(text="lowest corner", xy=(0, cdf["vout"].min()), offset=(12, -14), arrow=True)]
pwc.plot_corner_strip(cdf, metric="vout", target=5.0, annotations=ann_st,
                      path=FIG/"03_corner_strip_vout.png")
plt.show()
pwc.plot_corner_strip(cdf, metric="headroom", target=None,
                      path=FIG/"03_corner_strip_headroom.png")
plt.show()
pwc.plot_corner_strip(cdf, metric="p_diss", target=None,
                      path=FIG/"03_corner_strip_pdiss.png")
plt.show()

cell:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Tornado (OAT) — where to spend tolerance money

Each bar is the half-span with **everything else held nominal**. $V_{\mathrm{ADJ}}$
should dominate $V_{\mathrm{OUT}}$; $I_{\mathrm{ADJ}}$ is almost invisible at 2.2 kΩ.

In [7]:
for met, xlab, fname in (
    ("vout", "half-span (V)", "03_tornado_vout.png"),
    ("headroom", "half-span (V)", "03_tornado_headroom.png"),
    ("p_diss", "half-span (W)", "03_tornado_pdiss.png"),
    ("efficiency", "half-span", "03_tornado_eta.png"),
):
    plots.plot_tornado(oat[met], xlabel=xlab, title=f"OAT tornado — {met}",
                       path=FIG/fname)
    plt.show()
# Span grouped: envelope vs over-temp vs RSS
pwc.plot_span_grouped_bars(
    {
        "analytical": {
            "vout": wc.span("vout"),
            "headroom": wc.span("headroom"),
            "p_diss": wc.span("p_diss"),
            "efficiency": wc.span("efficiency"),
        },
    },
    path=FIG/"03_span_bars.png",
    title="Analytical WC span (25 °C envelope)",
    ylabel="span (native units)",
)
plt.show()

cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 25 °C vs over-temp envelope

MP grade is specified −55 °C to 125 °C. The over-temp $V_{\mathrm{ADJ}}$ box
(1.237–1.313 V) **includes** line and load, so it is the datasheet-faithful
hot/cold envelope — do not stack line/load on top of it.

In [8]:
pwc.plot_range_bars(
    {"analytical": pwc.band_from_wc(wc, "vout")},
    metric="vout", target=5.0, rss=(rss["vout"]["low"], rss["vout"]["high"]),
    annotations=[dict(text="25 °C envelope", xy=(wc.nominal["vout"], 0), offset=(10, 16), arrow=True)],
    path=FIG/"03_range_25C.png", title=r"$V_{\mathrm{OUT}}$ WC  25 °C envelope",
)
plt.show()
pwc.plot_range_bars(
    {"analytical": pwc.band_from_wc(wc_ot, "vout")},
    metric="vout", target=5.0,
    path=FIG/"03_range_overtemp.png", title=r"$V_{\mathrm{OUT}}$ WC  over-temp envelope (MP)",
)
plt.show()
pwc.plot_span_grouped_bars(
    {
        "25 °C": {m: wc.span(m) for m in ("vout", "headroom", "p_diss")},
        "over temp": {m: wc_ot.span(m) for m in ("vout", "headroom", "p_diss")},
        "RSS 25 °C": {m: 2 * rss[m]["delta_rss"] for m in ("vout", "headroom", "p_diss")},
    },
    path=FIG/"03_span_25_vs_ot.png",
    title="Span: 25 °C hypercube vs over-temp vs RSS",
)
plt.show()
plots.plot_wc_vout_vs_vin(p, annotations=[
    dict(text="25 °C envelope", xy=(12.0, analysis.envelope_bounds(p)["max"]),
         offset=(12, 10), arrow=True),
], path=FIG/"03_wc_vs_vin.png")
plt.show()
ann_span = [
    dict(text="5.000 V goal", xy=(5.0, 1.0), offset=(10, 16), arrow=True, color="#9467bd"),
]
plots.plot_wc_span_compare(
    [
        {"name": "25 °C", "min": wc.minimum["vout"], "max": wc.maximum["vout"],
         "nominal": wc.nominal["vout"]},
        {"name": "over temp", "min": wc_ot.minimum["vout"], "max": wc_ot.maximum["vout"],
         "nominal": wc_ot.nominal["vout"]},
        {"name": "RSS 25 °C", "min": rss["vout"]["low"], "max": rss["vout"]["high"],
         "nominal": wc.nominal["vout"]},
    ],
    metric=r"$V_{\mathrm{OUT}}$ (V)", title="WC span  25 °C vs over-temp vs RSS",
    target=5.0, annotations=ann_span, path=FIG/"03_wc_span_whiskers.png",
)
plt.show()

cell:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ngspice hypercube (same 16 corners)

Behavioral LDO OP at every analytical corner. Expect $V_{\mathrm{OUT}}$ within
a millivolt of the closed form. Set `SKIP_SPICE_WC_MC = True` to skip.

In [9]:
ng_df = None
HAVE_SPICE = False
try:
    find_ngspice()
    HAVE_SPICE = True
except FileNotFoundError as e:
    print(e)

if HAVE_SPICE and not SKIP_SPICE_WC_MC:
    rows = spice_wc_corners(
        p, workdir=ROOT/"results"/"ngspice_wc", max_workers=SPICE_WORKERS, full=True,
    )
    ng_df = pd.DataFrame(rows)
    display(ng_df[["corner", "vout_an", "vout_spice", "r1", "r2", "vadj"]])
    print("max |spice − an| mV", 1e3 * np.nanmax(np.abs(ng_df.vout_spice - ng_df.vout_an)))
    ng_band = pwc.band_from_array(ng_df["vout_spice"])
    pwc.plot_range_bars(
        {
            "analytical": pwc.band_from_wc(wc, "vout"),
            "ngspice": ng_band,
        },
        metric="vout", target=5.0, rss=(rss["vout"]["low"], rss["vout"]["high"]),
        annotations=[dict(text="ngspice vs closed form", xy=(ng_band[0], 0), offset=(10, 14), arrow=True)],
        path=FIG/"03_range_an_ng.png",
        title=r"$V_{\mathrm{OUT}}$ WC  analytical vs ngspice",
    )
    plt.show()
    pwc.plot_hypercube_scatter(
        cdf, x="headroom", y="vout", spice=ng_df.rename(columns={"vout_spice": "vout"}),
        nominal=(wc.nominal["headroom"], wc.nominal["vout"]), target_y=5.0,
        path=FIG/"03_hypercube_an_ng.png",
        title="Hypercube  analytical + ngspice",
    )
    plt.show()
    pwc.plot_corner_strip(
        cdf, metric="vout", spice=ng_df.rename(columns={"vout_spice": "vout"}),
        target=5.0, path=FIG/"03_strip_an_ng.png",
    )
    plt.show()
    pwc.plot_an_vs_spice(
        ng_df["vout_an"].to_numpy(), ng_df["vout_spice"].to_numpy(),
        engine="ngspice", metric="vout", path=FIG/"03_an_vs_ng.png",
    )
    plt.show()
    pwc.plot_whiskers_engines(
        {"analytical": pwc.band_from_wc(wc, "vout"), "ngspice": ng_band},
        metric="vout", target=5.0, path=FIG/"03_whiskers_an_ng.png",
    )
    plt.show()
else:
    print("ngspice WC skipped")

------------------------------------------------------------------------------


  ngspice WC hypercube


  ngspice  jobs=16  workers=8  requested=auto  cpu=8  env P5V_SPICE_WORKERS=''


  queue: R1lo_R2lo_Valo_Ialo, R1lo_R2lo_Valo_Iahi, R1lo_R2lo_Vahi_Ialo, R1lo_R2lo_Vahi_Iahi, R1lo_R2hi_Valo_Ialo, R1lo_R2hi_Valo_Iahi  +10 more


------------------------------------------------------------------------------


...

------------------------------------------------------------------------------


,corner,vout_an,vout_spice,r1,r2,vadj
0,R1lo_R2lo_Valo_Ialo,4.925,4.924951,0.0,9.900000e+11,4.925
1,R1lo_R2lo_Valo_Iahi,4.925,4.924951,0.0,9.900000e+11,4.925
2,R1lo_R2lo_Vahi_Ialo,5.075,5.074949,0.0,9.900000e+11,5.075
3,R1lo_R2lo_Vahi_Iahi,5.075,5.074949,0.0,9.900000e+11,5.075
4,R1lo_R2hi_Valo_Ialo,4.925,4.924951,0.0,1.010000e+12,4.925
5,R1lo_R2hi_Valo_Iahi,4.925,4.924951,0.0,1.010000e+12,4.925
6,R1lo_R2hi_Vahi_Ialo,5.075,5.074949,0.0,1.010000e+12,5.075
7,R1lo_R2hi_Vahi_Iahi,5.075,5.074949,0.0,1.010000e+12,5.075
8,R1hi_R2lo_Valo_Ialo,4.925,4.924951,0.0,9.900000e+11,4.925
9,R1hi_R2lo_Valo_Iahi,4.925,4.924951,0.0,9.900000e+11,4.925


max |spice − an| mV 0.05100000000002325


p5v_design/plots_wc_mc.py:109: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(9.6, max(2.6, 0.85 * len(clean) + 1.6)))
cell:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
